In [0]:
import os
import sys
import shutil
import json
from pathlib import Path

# Force local file system synchronization
os.sync()

# Dynamic workspace root directory resolution
ROOT_DIR = Path(os.getcwd()).parent
sys.path.append(str(ROOT_DIR))

# Define widgets for dynamic ClientContainer selection
dbutils.widgets.text("ClientContainer", "new", "Client Container / Catalog Name")
client_container = dbutils.widgets.get("ClientContainer").strip()

In [0]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from DimProvider.EDIProcessing.mapper import Mapper

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    
    # Clean destination if file already exists
    if target_path.exists():
        target_path.unlink()
        
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def process_single_file(incoming_file_path: Path, base_source_dir: Path, layout_id: str) -> tuple:
    """Parses an EDI file, extracts metadata, maps records, and generates a schema-compliant CSV."""
    active_file_path = incoming_file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        # Transition: pending -> inprogress
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Core Parsing & Domain Mapping
        structured_json = EDIProcessor().parse(str(active_file_path))
        
        # Metadata Extraction
        interchange = structured_json.get('interchange', {})
        client_id = interchange.get('sender_id', '02').strip()
        file_id = interchange.get('control_number', '01').strip()
        
        if not client_id:
            client_id = "02"
        if not file_id:
            file_id = "01"
            
        # Target CSV Location
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Code Conversion Execution
        CSVConverter().converter(Mapper().map_provider(structured_json), str(target_csv_path))
            
        print(f"client id: {client_id}, file id: {file_id}, layout id: {layout_id}, csv name: {target_csv_name}")
        return client_id, file_id, layout_id, target_csv_name, active_file_path
        
    except Exception as e:
        print(f"Failed processing {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise

In [0]:
def build_payloads(processed_files: list) -> str:
    """Generates precise production payloads for downstream bronze ingest stages."""
    process_list = []
    
    for f in processed_files:
        specific_client_container = f"{ROOT_DIR}/temp/{f['layout_id']}"
        schema_file = "provider_hierarchy_7.12_schema.json"
        dest_folder = "provider_hierarchy"
        
        process_list.append({
            "ClientID": f['client_id'],
            "FileID": f['file_id'],
            "FileName": f['csv_filename'],
            "ClientContainer": specific_client_container,
            "CurrentFolderPath": "",
            "ProcessedFolderPath": f"/Volumes/{client_container}/bronze/processed_parquet/{dest_folder}",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "SchemaFileName": schema_file,
            "SchemaFilePath": f"{ROOT_DIR}/DimProvider/Bronze/Schema",
            "TextQualifier": "\""
        })
        
    return json.dumps({"FileIds": process_list})

In [0]:
def trigger_silver_notebooks():
    """Triggers Silver layer notebooks in the correct dependency order."""
    silver_notebooks_base = f"{ROOT_DIR}/DimProvider/Silver/Notebooks"
    
    try:
        # Create Provider Hierarchy table (exclusively for 274 project)
        print("\n=== Triggering ProviderHierarchy (Silver Layer) ===")
        dbutils.notebook.run(f"{silver_notebooks_base}/ProviderHierarchy", 600)
        print("ProviderHierarchy completed successfully")
        
    except Exception as e:
        print(f"Silver layer processing failed: {e}")
        raise

In [0]:
def trigger_gold_notebooks(client_container_val: str):
    """Triggers Gold layer notebooks using the generic subgroup processor."""
    gold_notebooks_base = f"{ROOT_DIR}/DimProvider/Gold/Notebooks"
    
    try:
        # Load dimProvider (final merged dimension)
        print("\n=== Triggering dimProvider (Gold Layer) ===")
        dbutils.notebook.run(
            f"{gold_notebooks_base}/GenericSubGroupProcessing", 
            600, 
            {
                "ClientContainer": client_container_val,
                "SubGroupConfigPath": f"{ROOT_DIR}/DimProvider/Gold/Schema/dimProvider.json"
            }
        )
        print("dimProvider Gold load completed successfully")
        
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [0]:
def main():
    # Detect pending EDI files
    base_274_dir = ROOT_DIR / "source/274"
    pending_274_dir = base_274_dir / "pending"
    
    processed_files = []
    
    # Check for 274 (Provider Hierarchy) pending files
    if pending_274_dir.exists():
        incoming_274 = [f for f in pending_274_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
        for file_path in incoming_274:
            try:
                c_id, f_id, l_id, csv_filename, active_path = process_single_file(file_path, base_274_dir, "274")
                processed_files.append({
                    'client_id': c_id, 
                    'file_id': f_id, 
                    'layout_id': l_id,
                    'csv_filename': csv_filename,
                    'active_file_path': active_path
                })
            except Exception as e:
                print(f"Skipping 274 file {file_path.name}: {e}")

    if not processed_files:
        print("No pending EDI files found to process. Proceeding directly to Silver and Gold layers.")
        # Trigger Silver and Gold layers using existing Bronze data
        try:
            trigger_silver_notebooks()
            trigger_gold_notebooks(client_container)
            print("\nPipeline execution sequence completed successfully (Silver → Gold).")
        except Exception as e:
            print(f"Pipeline execution failed: {e}")
            raise
        return

    # If there are new CSVs, trigger Bronze Ingestion
    process_payload = build_payloads(processed_files)
    notebook_base = f"{ROOT_DIR}/Shared/Notebooks"
    
    try:
        print("\n=== Triggering FilesToProcess (Bronze Layer) ===")
        dbutils.notebook.run(f"{notebook_base}/FilesToProcess", 600, {"ProcessedJSON": process_payload})
        print("FilesToProcess completed successfully")
        
        # Trigger Silver layer notebooks
        trigger_silver_notebooks()
        
        # Trigger Gold layer notebooks
        trigger_gold_notebooks(client_container)
        
        # Transition: inprogress -> processed ONLY after complete success
        for f in processed_files:
            move_file(f['active_file_path'], base_274_dir / "processed")
        
        print("\nPipeline execution sequence completed successfully (Bronze → Silver → Gold).")
    except Exception as e:
        print(f"Downstream orchestration failed: {e}")
        # Transition: inprogress -> failed on downstream failure
        for f in processed_files:
            if f['active_file_path'].exists():
                move_file(f['active_file_path'], base_274_dir / "failed")
        raise

if __name__ == "__main__":
    main()